# 🎨 PhotoEnhance — AI Photo Upscaler

**Покращення якості фото за допомогою Real-ESRGAN — один запуск, одна кнопка.**

| Параметр | Значення |
|---|---|
| Модель | Real-ESRGAN x2 / x4 / x8 |
| GPU | Google Colab T4 (16 GB) |
| Формати | JPG, PNG, WEBP |
| Вартість | $0 |

> ⚠️ Увімкни GPU: **Runtime → Change runtime type → T4 GPU**
>
> Потім натисни **▶ Run** на клітинці нижче і чекай — все налаштується автоматично.


In [ ]:
# @title 🎨 PhotoEnhance — Один крок { display-mode: "form" }
# Запусти один раз. Перший запуск: ~2 хв на встановлення.
# Повторний запуск: миттєво (завантажено в кеш).

import sys, os, io, re, base64, time, shutil, threading, subprocess, urllib.request
from types import ModuleType

import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ═══════════════════════════════════════════════════════
# §0  ВСТАНОВЛЕННЯ (автоматично, якщо потрібно)
# ═══════════════════════════════════════════════════════
_pb_setup = widgets.IntProgress(
    value=0, min=0, max=100, description='⏳  0%',
    bar_style='info',
    style={'bar_color': '#4a90e2', 'description_width': '65px'},
    layout=widgets.Layout(width='100%', height='28px'),
)
_lbl_setup = widgets.HTML(
    '<span style="font-size:12px;color:#555;">Перевірка залежностей…</span>'
)
display(widgets.VBox([_pb_setup, _lbl_setup]))

_needs = False
try:
    import realesrgan as _re_chk; del _re_chk
    import cv2 as _cv_chk;        del _cv_chk
except ImportError:
    _needs = True

if _needs:
    os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
    os.environ['PIP_NO_WARN_SCRIPT_LOCATION']   = '1'

    def _pip(args):
        get_ipython().run_line_magic('pip', args)  # type: ignore

    _lbl_setup.value = '<span style="font-size:12px;color:#555;">📦 basicsr / gfpgan…</span>'
    _pip('install -q "basicsr==1.4.2" facexlib gfpgan')
    _pb_setup.value = 20; _pb_setup.description = '📦 20%'

    _lbl_setup.value = '<span style="font-size:12px;color:#555;">📦 Pillow / opencv / ipywidgets…</span>'
    _pip('install -q "Pillow>=9" "opencv-python-headless>=4.8" ipywidgets')
    _pb_setup.value = 35; _pb_setup.description = '📦 35%'

    if not os.path.isdir('/content/Real-ESRGAN'):
        _lbl_setup.value = '<span style="font-size:12px;color:#555;">📦 Клонування Real-ESRGAN…</span>'
        get_ipython().system('git clone -q https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN')  # type: ignore
    _pb_setup.value = 55; _pb_setup.description = '📦 55%'

    _lbl_setup.value = '<span style="font-size:12px;color:#555;">📦 pip install Real-ESRGAN…</span>'
    _pip('install -q -e /content/Real-ESRGAN')
    _pb_setup.value = 75; _pb_setup.description = '📦 75%'

    # re-pin basicsr — requirements.txt може підтягнути новішу
    _pip('install -q "basicsr==1.4.2"')
    print('✅ basicsr==1.4.2 OK')

# torchvision compat patch (basicsr шукає старий module)
import torchvision.transforms.functional as _tvf
_cm = ModuleType('torchvision.transforms.functional_tensor')
_cm.rgb_to_grayscale = _tvf.rgb_to_grayscale  # type: ignore
sys.modules.setdefault('torchvision.transforms.functional_tensor', _cm)

for _p in ['/content/Real-ESRGAN']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

_pb_setup.value = 80; _pb_setup.description = '🔌 80%'

import torch
torch.backends.cudnn.benchmark        = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
except AttributeError:
    pass

USE_FP16 = torch.cuda.is_available()

# KeepAlive — щоб Colab не відключив сесію
display(Javascript("""
(function(){
  if (window._pe_ka) clearInterval(window._pe_ka);
  window._pe_ka = setInterval(function(){ fetch('/api/kernels').catch(function(){}); }, 45000);
})();
"""))
threading.Thread(
    target=lambda: threading.Event().wait(timeout=3600),
    daemon=True, name='PE-KA'
).start()

_pb_setup.value = 90; _pb_setup.description = '🖥️ 90%'

# GPU info
if USE_FP16:
    _gn    = torch.cuda.get_device_name(0)
    _gmem  = torch.cuda.get_device_properties(0).total_memory / 1073741824
    _cc    = torch.cuda.get_device_capability(0)
    _gpu_s = f'GPU: {_gn}  ({_gmem:.1f} GB)  Compute {_cc[0]}.{_cc[1]}'
    import subprocess as _sp
    _nsmi = _sp.run(
        ['nvidia-smi', '--query-gpu=memory.free,temperature.gpu', '--format=csv,noheader,nounits'],
        capture_output=True, text=True,
    )
    if _nsmi.returncode == 0:
        _pts = _nsmi.stdout.strip().split(', ')
        try:
            _gpu_s += f'  |  free: {int(_pts[0])/1024:.1f} GB  {_pts[1]}°C'
        except Exception:
            pass
else:
    _gpu_s = '⚠️ GPU не знайдено — CPU (дуже повільно)'

print(f'✅ {_gpu_s}')
print(f'✅ PyTorch {torch.__version__}  |  {"FP16+TF32" if USE_FP16 else "FP32"}')

_pb_setup.value = 100; _pb_setup.description = '✅ 100%'; _pb_setup.bar_style = 'success'
_lbl_setup.value = (
    '<span style="font-size:12px;color:#2ecc71;font-weight:bold;">'
    '✅ Готово! Завантажуй фото нижче ↓</span>'
)

# ═══════════════════════════════════════════════════════
# §1  ESRGAN — кеш моделей
# ═══════════════════════════════════════════════════════
if '_PE_CACHE' not in globals():
    _PE_CACHE: dict = {}

_MODELS_DIR = '/content/pe_models'
os.makedirs(_MODELS_DIR, exist_ok=True)

_MODEL_URLS = {
    'RealESRGAN_x4plus': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth', 4),
    'RealESRGAN_x2plus': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth', 2),
    'RealESRGAN_x4plus_anime_6B': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth', 4),
}

import cv2
cv2.setNumThreads(0)
from PIL import Image as _PIL_Image
_PIL_Image.MAX_IMAGE_PIXELS = None

def _get_esrgan(name: str, h: int = 0, w: int = 0):
    from realesrgan             import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet

    mpx  = h * w / 1_000_000
    tile = 0 if mpx < 2 else (512 if mpx < 8 else 256)
    key  = f'{name}_{USE_FP16}'

    if key in _PE_CACHE:
        m = _PE_CACHE[key]
        m.tile = tile; m.tile_pad = 32 if tile else 0
        print(f'♻️  кеш: {name}  tile={tile or "вимк"}')
        return m

    url, scale  = _MODEL_URLS[name]
    model_path  = os.path.join(_MODELS_DIR, f'{name}.pth')
    if not os.path.isfile(model_path):
        print(f'⬇️  Завантаження {name}…')
        urllib.request.urlretrieve(url, model_path)

    n_block = 6 if 'anime_6B' in name else 23
    rrd     = RRDBNet(
        num_in_ch=3, num_out_ch=3, num_feat=64,
        num_block=n_block, num_grow_ch=32, scale=scale,
    )
    m = RealESRGANer(
        scale=scale, model_path=model_path, model=rrd,
        tile=tile, tile_pad=32 if tile else 0, pre_pad=0,
        half=USE_FP16,
    )
    m.model.eval()

    # warmup
    try:
        _dev = torch.device('cuda' if USE_FP16 else 'cpu')
        _d   = torch.zeros(1, 3, 64, 64, device=_dev)
        if USE_FP16: _d = _d.half()
        with torch.inference_mode(): m.model(_d)
        if USE_FP16: torch.cuda.synchronize()
        print('🔥 Warmup OK')
    except Exception:
        pass

    _PE_CACHE[key] = m
    print(f'✅ {name}  tile={tile or "вимк"}  FP{"16" if USE_FP16 else "32"}')
    return m


def _run_esrgan(in_path: str, out_path: str, model_name: str, outscale: int, pb) -> tuple:
    img = cv2.imread(in_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f'Не вдалося відкрити: {in_path}')
    h, w = img.shape[:2]
    up   = _get_esrgan(model_name, h, w)

    _real = sys.stdout
    class _Prog:
        def write(self, t: str) -> None:
            m = re.search(r'Tile\s+(\d+)/(\d+)', t)
            if m:
                pct = 20 + int(int(m.group(1)) / int(m.group(2)) * 75)
                pb.value = min(pct, 95); pb.description = f'🔄 {min(pct,95):>3}%'
        def flush(self)              -> None: _real.flush()
        def __getattr__(self, n: str): return getattr(_real, n)

    sys.stdout = _Prog()
    try:
        out_img, _ = up.enhance(img, outscale=outscale)
    except RuntimeError as exc:
        sys.stdout = _real
        if 'out of memory' in str(exc).lower():
            torch.cuda.empty_cache()
            up.tile = 128; up.tile_pad = 16
            print('⚠️  OOM → tile=128 retry…')
            sys.stdout = _Prog()
            out_img, _ = up.enhance(img, outscale=outscale)
            sys.stdout = _real
        else:
            raise
    except Exception:
        sys.stdout = _real; raise
    else:
        sys.stdout = _real

    if USE_FP16: torch.cuda.synchronize()
    h2, w2 = out_img.shape[:2]
    fl = [cv2.IMWRITE_JPEG_QUALITY, 95] if out_path.lower().endswith(('.jpg', '.jpeg')) else []
    cv2.imwrite(out_path, out_img, fl)
    del out_img; torch.cuda.empty_cache()
    return (w, h), (w2, h2)


# ═══════════════════════════════════════════════════════
# §2  UI
# ═══════════════════════════════════════════════════════
TEMP_DIR = '/content/photoenhance_temp'
os.makedirs(TEMP_DIR, exist_ok=True)
MAX_MB   = 100

display(HTML("""
<style>
@media(max-width:600px){
  .widget-upload,.widget-button{width:100%!important;min-height:52px!important;font-size:16px!important;}
  .widget-toggle-buttons .widget-toggle-button{font-size:12px!important;padding:5px 8px!important;}
}
</style>
"""))

_w_title = widgets.HTML("""
<div style="font-family:sans-serif;margin:8px 0;">
  <h2 style="margin:0 0 4px;">🎨 PhotoEnhance</h2>
  <p style="color:#666;margin:0;font-size:13px;">Real-ESRGAN | JPG / PNG / WEBP | Макс %(max)s MB</p>
</div>
""" % {'max': MAX_MB})

_w_upload = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.webp', multiple=False,
    description='📸 Вибрати фото',
    layout=widgets.Layout(width='auto', min_width='200px', min_height='44px'),
)

_w_scale = widgets.ToggleButtons(
    options=[('×2  (швидко)', 2), ('×4  (оптимально)', 4), ('×8  (макс розмір)', 8)],
    value=4, description='Масштаб:',
    style={'button_width': 'auto', 'description_width': '70px'},
    layout=widgets.Layout(width='100%'),
)

_w_model = widgets.Dropdown(
    options=[
        ('🖼  Загальні фото',    'RealESRGAN_x4plus'),
        ('🎌 Аніме / Ілюстрації', 'RealESRGAN_x4plus_anime_6B'),
    ],
    value='RealESRGAN_x4plus', description='Стиль:',
    layout=widgets.Layout(width='280px'),
)

_w_btn = widgets.Button(
    description='⚡ Покращити фото',
    button_style='success',
    layout=widgets.Layout(width='220px', height='46px'),
)
_w_btn.style.font_size = '15px'

_w_info    = widgets.HTML('<div style="font-size:13px;color:#777;">📁 Фото не вибрано</div>')
_w_preview = widgets.Output()
_w_result  = widgets.Output()


def _thumb_show(img_pil, title=''):
    th = img_pil.copy(); th.thumbnail((260, 260))
    import matplotlib.pyplot as _plt
    _, ax = _plt.subplots(figsize=(3, 3))
    ax.imshow(th); ax.axis('off')
    if title: ax.set_title(title, fontsize=9)
    _plt.tight_layout(); _plt.show()


def _on_upload(*_):
    if not _w_upload.value:
        return
    for fn, fd in _w_upload.value.items():
        sz = len(fd['content']) / 1048576
        if sz > MAX_MB:
            _w_info.value = (
                f'<div style="color:red;font-size:13px;">❌ Файл {sz:.1f} MB > ліміту {MAX_MB} MB.</div>'
            )
            return
        try:
            img = _PIL_Image.open(io.BytesIO(fd['content']))
            w, h = img.size
            _w_info.value = (
                f'<div style="font-size:13px;color:#1a8c1a;">'
                f'✅ <b>{fn}</b> | {w}×{h} px | {sz:.1f} MB | {img.mode}'
                f'</div>'
            )
            with _w_preview:
                clear_output(wait=True)
                _thumb_show(img, 'Оригінал')
        except Exception as e:
            _w_info.value = f'<div style="color:red;font-size:13px;">❌ {e}</div>'

_w_upload.observe(_on_upload, names='value')


def _pil_b64(img: _PIL_Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=88)
    return base64.b64encode(buf.getvalue()).decode()


def _dl_link(path: str, label: str, mime: str = 'image/jpeg') -> str:
    with open(path, 'rb') as fh:
        b64 = base64.b64encode(fh.read()).decode()
    fn = os.path.basename(path)
    return (
        f'<a href="data:{mime};base64,{b64}" download="{fn}" '
        f'style="display:inline-block;margin:6px 4px;padding:11px 20px;'
        f'background:#2ecc71;color:#fff;font-weight:bold;font-size:14px;'
        f'border-radius:8px;text-decoration:none;">⬇️ {label}</a>'
    )


def _on_enhance(_):
    if not _w_upload.value:
        _w_info.value = '<div style="color:red;font-size:13px;">❌ Спочатку вибери фото!</div>'
        return

    with _w_result:
        clear_output(wait=True)

        fn  = list(_w_upload.value.keys())[0]
        fd  = list(_w_upload.value.values())[0]
        raw = fd['content']
        sz  = len(raw) / 1048576

        if sz > MAX_MB:
            print(f'❌ Файл {sz:.1f} MB > {MAX_MB} MB'); return
        if sz == 0:
            print('❌ Файл порожній'); return

        # визначаємо реальний масштаб і модель
        outscale    = _w_scale.value
        sel_model   = _w_model.value

        # для x2 — спеціальна x2plus модель; аніме лише x4
        if sel_model == 'RealESRGAN_x4plus_anime_6B':
            model_name = 'RealESRGAN_x4plus_anime_6B'
            outscale   = min(outscale, 4)   # аніме модель — тільки x4 max
        elif outscale == 2:
            model_name = 'RealESRGAN_x2plus'
        else:
            model_name = 'RealESRGAN_x4plus'

        # перевірка розміру виходу
        try:
            _pil_chk     = _PIL_Image.open(io.BytesIO(raw))
            _iw, _ih     = _pil_chk.size
            _out_mpx     = _iw * _ih * outscale ** 2 / 1_000_000
            if _out_mpx > 200:
                print(
                    f'❌ Вихід {_out_mpx:.0f} Mpx — завеликий! '
                    f'Зменш масштаб або обріж фото (зараз {_iw}×{_ih}).'
                )
                return
        except Exception:
            pass

        # VRAM
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            _free = (
                torch.cuda.get_device_properties(0).total_memory
                - torch.cuda.memory_allocated()
            ) / 1073741824
            if _free < 1.5:
                print(f'❌ Мало VRAM ({_free:.1f} GB). Runtime → Restart runtime.'); return

        # зберігаємо вхідний файл
        in_path  = os.path.join(TEMP_DIR, fn)
        base     = os.path.splitext(fn)[0]
        out_ext  = 'png' if _out_mpx <= 80 else 'jpg'
        out_name = f'{base}_x{outscale}.{out_ext}'
        out_path = os.path.join(TEMP_DIR, out_name)

        with open(in_path, 'wb') as fh:
            fh.write(raw)

        # ── прогрес ──
        pb = widgets.IntProgress(
            value=5, min=0, max=100, description='⬇️   5%',
            bar_style='info',
            style={'bar_color': '#4a90e2', 'description_width': '65px'},
            layout=widgets.Layout(width='100%', height='28px'),
        )
        display(pb)
        _fp_lbl = 'FP16+TF32' if USE_FP16 else 'FP32'
        print(f'🎨 {fn}  |  {model_name}  |  ×{outscale}  |  {_fp_lbl}')

        # ── enhance ──
        t0 = time.time()
        try:
            orig_sz, out_sz = _run_esrgan(in_path, out_path, model_name, outscale, pb)
        except Exception as exc:
            pb.bar_style = 'danger'; pb.description = '❌ Помилка'
            print(f'❌ {exc}'); raise

        pb.value = 100; pb.description = '✅ 100%'; pb.bar_style = 'success'
        elapsed = time.time() - t0
        px_mult = (out_sz[0] * out_sz[1]) / max(1, orig_sz[0] * orig_sz[1])
        print(
            f'\n✅ {elapsed:.1f} сек  |  '
            f'{orig_sz[0]}×{orig_sz[1]} → {out_sz[0]}×{out_sz[1]}  |  '
            f'×{px_mult:.1f} пікс  |  {_fp_lbl}'
        )

        # ── BEFORE / AFTER слайдер ──
        orig_pil = _PIL_Image.open(in_path).convert('RGB')
        enh_pil  = _PIL_Image.open(out_path).convert('RGB')
        _MD = 1100
        orig_s = orig_pil.copy(); orig_s.thumbnail((_MD, _MD), _PIL_Image.LANCZOS)
        enh_s  = enh_pil.copy();  enh_s.thumbnail((_MD, _MD),  _PIL_Image.LANCZOS)

        b64_orig = _pil_b64(orig_s)
        b64_enh  = _pil_b64(enh_s)

        _oninput = (
            "var p=this.value;var par=this.parentElement;"
            "par.querySelector('.pe-bef').style.clipPath='inset(0 '+(100-p)+'% 0 0)';"
            "par.querySelector('.pe-line').style.left=p+'%';"
        )
        display(HTML(
            '<div style="position:relative;display:inline-block;max-width:100%;'
            'user-select:none;-webkit-user-select:none;overflow:hidden;">'
            f'<img src="data:image/jpeg;base64,{b64_enh}"'
            ' style="display:block;max-width:100%;height:auto;pointer-events:none;">'
            f'<img class="pe-bef" src="data:image/jpeg;base64,{b64_orig}"'
            ' style="position:absolute;top:0;left:0;width:100%;'
            'clip-path:inset(0 50% 0 0);pointer-events:none;">'
            '<div class="pe-line" style="position:absolute;top:0;left:50%;'
            'width:3px;height:100%;background:#fff;'
            'box-shadow:0 0 6px rgba(0,0,0,.6);z-index:10;'
            'transform:translateX(-50%);pointer-events:none;">'
            '<div style="position:absolute;top:50%;left:50%;'
            'transform:translate(-50%,-50%);width:40px;height:40px;'
            'border-radius:50%;background:#fff;display:flex;align-items:center;'
            'justify-content:center;font-size:20px;font-family:sans-serif;'
            'box-shadow:0 2px 8px rgba(0,0,0,.5);">&#8596;</div></div>'
            '<div style="position:absolute;bottom:8px;left:8px;'
            'background:rgba(0,0,0,.6);color:#fff;font-size:12px;'
            'padding:2px 8px;border-radius:4px;pointer-events:none;'
            'font-family:sans-serif;">BEFORE</div>'
            '<div style="position:absolute;bottom:8px;right:8px;'
            'background:rgba(0,0,0,.6);color:#fff;font-size:12px;'
            'padding:2px 8px;border-radius:4px;pointer-events:none;'
            'font-family:sans-serif;">AFTER</div>'
            '<input type="range" min="0" max="100" value="50"'
            ' style="position:absolute;top:0;left:0;width:100%;height:100%;'
            'opacity:0;cursor:ew-resize;margin:0;padding:0;'
            '-webkit-appearance:none;appearance:none;z-index:20;"'
            f' oninput="{_oninput}"></div>'
            '<p style="font-size:12px;color:#888;margin:4px 0 0;">'
            '&#8592; Потягни для порівняння &nbsp;|&nbsp; BEFORE ліво / AFTER право</p>'
        ))

        # ── side-by-side для скачування ──
        cmp_path = os.path.join(TEMP_DIR, 'comparison.jpg')
        cw = orig_s.width + enh_s.width
        ch = max(orig_s.height, enh_s.height)
        cmp = _PIL_Image.new('RGB', (cw, ch), (26, 26, 46))
        cmp.paste(orig_s, (0, 0)); cmp.paste(enh_s, (orig_s.width, 0))
        cmp.save(cmp_path, 'JPEG', quality=92)

        out_mime = 'image/png' if out_ext == 'png' else 'image/jpeg'
        display(HTML(
            '<div style="margin-top:10px;">'
            + _dl_link(out_path,  'Покращений', out_mime)
            + _dl_link(cmp_path, 'Before / After', 'image/jpeg')
            + '</div>'
        ))

        # auto-download (Colab)
        try:
            from google.colab import files as _cf  # type: ignore
            _cf.download(out_path)
        except Exception:
            pass

        print('\n✅ Готово!')


_w_btn.on_click(_on_enhance)

# ── зібрати UI ──────────────────────────────────────
_ui = widgets.VBox([
    _w_title,
    widgets.HTML('<hr style="margin:6px 0">'),
    _w_upload, _w_info, _w_preview,
    widgets.HTML('<hr style="margin:6px 0">'),
    _w_scale,
    _w_model,
    widgets.HTML('<hr style="margin:6px 0">'),
    _w_btn,
    _w_result,
], layout=widgets.Layout(width='100%', max_width='660px'))

display(_ui)